In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import issparse

import scanpy as sc

import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax.ops import segment_sum

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
import optax


# -----------------------------
# 1) Build a tidy DataFrame from AnnData
# -----------------------------

def make_casecontrol_df(
    adata,
    feature: str,
    condition_key: str = "condition",
    sample_key: str = "sample_id",
    section_key: str = "sample_id",
    group_key: str = "cell_class",
) -> pd.DataFrame:
    """
    Extract a single feature (gene/lipid) and relevant obs metadata
    into a tidy DataFrame.

    Columns:
      - feature (one column)
      - Condition
      - Sample
      - SectionID
      - group
    """
    if feature not in adata.var_names:
        raise ValueError(f"{feature} not found in adata.var_names")

    for k in [condition_key, sample_key, section_key, group_key]:
        if k not in adata.obs.columns:
            raise ValueError(f"Obs column '{k}' is missing from adata.obs")

    # Robust extraction of the feature vector (handles dense & sparse)
    mat = adata[:, feature].X
    if issparse(mat):
        values = mat.toarray().ravel().astype(np.float32)
    else:
        values = np.asarray(mat).ravel().astype(np.float32)

    df = pd.DataFrame({feature: values}, index=adata.obs_names)
    df["Condition"] = adata.obs[condition_key].values
    df["Sample"] = adata.obs[sample_key].values
    df["SectionID"] = adata.obs[section_key].values
    df["group"] = adata.obs[group_key].values
    return df


# -----------------------------
# 2) Encode design matrices
# -----------------------------

def make_design_matrices(df: pd.DataFrame) -> dict:
    """
    From a tidy DF with columns:
      Condition, Sample, SectionID, group
    build integer codes and mappings needed by the model.
    """

    # Encode as categorical codes
    for col, code_name in [
        ("Condition", "condition_code"),
        ("Sample", "sample_code"),
        ("SectionID", "section_code"),
        ("group", "group_code"),
    ]:
        df[code_name] = df[col].astype("category").cat.codes.astype(int)

    # Map from section -> sample
    sec2samp = (
        df[["section_code", "sample_code"]]
        .drop_duplicates()
        .sort_values("section_code")
    )
    section2sample = sec2samp["sample_code"].values.astype(int)

    # Map from sample -> condition
    samp2cond = (
        df[["sample_code", "condition_code"]]
        .drop_duplicates()
        .sort_values("sample_code")
    )
    sample2condition = samp2cond["condition_code"].values.astype(int)

    # Basic counts
    n_groups = int(df["group_code"].max() + 1)
    n_samples = int(df["sample_code"].max() + 1)
    n_sections = int(df["section_code"].max() + 1)
    n_conditions = int(df["condition_code"].nunique())
    n_obs = int(df.shape[0])

    if n_conditions != 2:
        raise ValueError(
            f"Model is currently implemented for exactly 2 conditions, "
            f"but found {n_conditions} in 'Condition'."
        )

    design = dict(
        condition_code=df["condition_code"].values.astype(int),
        section_code=df["section_code"].values.astype(int),
        group_code=df["group_code"].values.astype(int),
        section2sample=section2sample,
        sample2condition=sample2condition,
        n_groups=n_groups,
        n_samples=n_samples,
        n_sections=n_sections,
        n_conditions=n_conditions,
        n_obs=n_obs,
    )
    return design


# -----------------------------
# 3) Hierarchical case–control model (JAX/Numpyro)
# -----------------------------

def hierarchical_casecontrol_model(
    condition_code,
    section_code,
    group_code,
    section2sample,
    sample2condition,
    n_groups: int,
    n_samples: int,
    n_sections: int,
    n_conditions: int,
    n_obs: int,
    y=None,
    supertype_prior_sd: float = 1.0,
    supertype_shift_prior_sd: float = 1.0,
    sample_prior_sd: float = 1.0,
    section_prior_sd: float = 5.0,
    obs_sd: float = 0.1,
):
    """
    Hierarchical Gaussian model:

    y_{i} = section_effect[section_i]
            + base_group[group_i]
            + shift_group[group_i] * I[condition_i == 1]
            + eps_i

    with:
      - group-level base & shift
      - sample-level random effects
      - section-level random effects (centered by condition to absorb batch)
    """

    # ------------------------
    # Group-level (e.g. cell_class)
    # ------------------------
    with numpyro.plate("group_plate", n_groups):
        base_group = numpyro.sample(
            "base_group",
            dist.Normal(0.0, supertype_prior_sd),
        )
        shift_group = numpyro.sample(
            "shift_group",
            dist.Normal(0.0, supertype_shift_prior_sd),
        )

    # ------------------------
    # Sample-level
    # ------------------------
    with numpyro.plate("sample_plate", n_samples):
        mu_sample = numpyro.sample(
            "mu_sample",
            dist.Normal(0.0, sample_prior_sd),
        )
        log_sigma_sample = numpyro.sample(
            "log_sigma_sample",
            dist.Normal(0.0, section_prior_sd),
        )
    sigma_sample = jnn.softplus(log_sigma_sample)

    # ------------------------
    # Section-level (non-centered)
    # ------------------------
    with numpyro.plate("section_plate", n_sections):
        eps_section = numpyro.sample("eps_section", dist.Normal(0.0, 1.0))
        mu_sec = mu_sample[section2sample]          # [n_sections]
        sigma_sec = sigma_sample[section2sample]    # [n_sections]
        section_raw = mu_sec + eps_section * sigma_sec

    # Center section-level effects by condition (to break identifiability)
    cond_per_section = sample2condition[section2sample]  # shape [n_sections]

    sum_by_cond = segment_sum(section_raw, cond_per_section, n_conditions)
    count_by_cond = segment_sum(
        jnp.ones_like(section_raw), cond_per_section, n_conditions
    )
    mean_by_cond = sum_by_cond / count_by_cond

    section_effect = section_raw - mean_by_cond[cond_per_section]

    # ------------------------
    # Linear predictor per observation i
    # ------------------------
    mu = (
        section_effect[section_code]
        + base_group[group_code]
        + jnp.where(condition_code == 1, shift_group[group_code], 0.0)
    )

    # ------------------------
    # Observation model
    # ------------------------
    with numpyro.plate("obs", n_obs):
        numpyro.sample("y", dist.Normal(mu, obs_sd), obs=y)


# -----------------------------
# 4) Fit model for a single feature
# -----------------------------

def fit_casecontrol_single(
    df: pd.DataFrame,
    feature: str,
    num_steps: int = 1500,
    lr: float = 0.03,
    seed: int = 0,
    priors: dict | None = None,
):
    """
    Fit the hierarchical case–control model to a single feature column in df.

    Returns:
      stats_df : DataFrame with posterior stats per group (e.g. cell_class)
      samples  : dict of posterior samples for 'shift_group'
      params   : learned variational parameters
    """
    if priors is None:
        priors = {}

    design = make_design_matrices(df)
    y = df[feature].values.astype(np.float32)

    # Convert design arrays to jnp
    cc = jnp.array(design["condition_code"])
    scode = jnp.array(design["section_code"])
    gcode = jnp.array(design["group_code"])
    s2s = jnp.array(design["section2sample"])
    s2c = jnp.array(design["sample2condition"])
    n_groups = design["n_groups"]
    n_samples = design["n_samples"]
    n_sections = design["n_sections"]
    n_conditions = design["n_conditions"]
    n_obs = design["n_obs"]

    # Wrap model so we can pass priors cleanly
    def model_wrapper(
        condition_code,
        section_code,
        group_code,
        section2sample,
        sample2condition,
        n_groups,
        n_samples,
        n_sections,
        n_conditions,
        n_obs,
        y=None,
    ):
        return hierarchical_casecontrol_model(
            condition_code=condition_code,
            section_code=section_code,
            group_code=group_code,
            section2sample=section2sample,
            sample2condition=sample2condition,
            n_groups=n_groups,
            n_samples=n_samples,
            n_sections=n_sections,
            n_conditions=n_conditions,
            n_obs=n_obs,
            y=y,
            **priors,
        )

    guide = AutoNormal(model_wrapper)
    optimizer = optax.adam(lr)
    svi = SVI(model_wrapper, guide, optimizer, loss=Trace_ELBO())

    rng_key = jax.random.PRNGKey(seed)
    svi_state = svi.init(
        rng_key,
        condition_code=cc,
        section_code=scode,
        group_code=gcode,
        section2sample=s2s,
        sample2condition=s2c,
        n_groups=n_groups,
        n_samples=n_samples,
        n_sections=n_sections,
        n_conditions=n_conditions,
        n_obs=n_obs,
        y=jnp.array(y),
    )

    for step in range(num_steps):
        svi_state, loss = svi.update(
            svi_state,
            condition_code=cc,
            section_code=scode,
            group_code=gcode,
            section2sample=s2s,
            sample2condition=s2c,
            n_groups=n_groups,
            n_samples=n_samples,
            n_sections=n_sections,
            n_conditions=n_conditions,
            n_obs=n_obs,
            y=jnp.array(y),
        )
        if (step + 1) % 200 == 0:
            print(f"  step {step+1}/{num_steps}, ELBO={loss:.3f}")

    params = svi.get_params(svi_state)

    # Posterior samples for shift_group (per group)
    predictive = Predictive(
        model_wrapper,
        guide=guide,
        params=params,
        num_samples=1000,
        return_sites=["shift_group"],
    )
    samples = predictive(
        rng_key,
        condition_code=cc,
        section_code=scode,
        group_code=gcode,
        section2sample=s2s,
        sample2condition=s2c,
        n_groups=n_groups,
        n_samples=n_samples,
        n_sections=n_sections,
        n_conditions=n_conditions,
        n_obs=n_obs,
        y=None,
    )

    shift_samples = np.array(samples["shift_group"])  # shape [num_draws, n_groups]

    # Basic posterior summary + Bayesian FDR
    loc = shift_samples.mean(axis=0)
    sd = shift_samples.std(axis=0)
    p_pos = (shift_samples > 0).mean(axis=0)
    p_neg = (shift_samples < 0).mean(axis=0)
    PEP = np.minimum(p_pos, p_neg)  # posterior error probability

    order = np.argsort(PEP)
    cum_PEP = np.cumsum(PEP[order]) / np.arange(1, len(PEP) + 1)
    qvalue = np.empty_like(PEP)
    qvalue[order] = cum_PEP

    # map group_code -> group name
    mapping = (
        df[["group", "group_code"]]
        .drop_duplicates()
        .sort_values("group_code")
    )
    group_names = mapping["group"].values

    stats_df = pd.DataFrame(
        {
            "posterior_mean": loc,
            "posterior_sd": sd,
            "p(>0)": p_pos,
            "p(<0)": p_neg,
            "PEP": PEP,
            "qvalue": qvalue,
            "selected_fdr_0.05": qvalue < 0.05,
        },
        index=group_names,
    ).sort_values("qvalue")

    return stats_df, samples, params


# -----------------------------
# 5) High-level: run on several features directly from AnnData
# -----------------------------

def run_casecontrol_on_adata(
    adata,
    features,
    condition_key="condition",
    sample_key="sample_id",
    section_key="sample_id",
    group_key="cell_class",
    num_steps=1500,
    lr=0.03,
    seed=0,
):
    """
    Convenience wrapper:
      - pulls data from AnnData
      - fits hierarchical case–control model for each feature
      - returns dict {feature -> stats_df}
    """
    results = {}
    for f in features:
        print(f"\n=== Fitting case–control model for {f} ===")
        df = make_casecontrol_df(
            adata,
            feature=f,
            condition_key=condition_key,
            sample_key=sample_key,
            section_key=section_key,
            group_key=group_key,
        )
        stats_f, _, _ = fit_casecontrol_single(
            df,
            feature=f,
            num_steps=num_steps,
            lr=lr,
            seed=seed,
        )
        results[f] = stats_f
    return results

/Users/christoffer/miniconda3/envs/EUCLID_ENV/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
import os
base_dir = '../results/casecontrol_hvg_results_live_condition/'
df_ = []
files = os.listdir(base_dir)
for file in files:
    if file!='.ipynb_checkpoints':
        casecontrol = pd.read_csv(base_dir + file)
        df_.append(casecontrol)    
concat = pd.concat(df_)
concat = concat[concat['Unnamed: 0'] != 'Myocytes']

concat = concat.sort_values(by = 'posterior_mean', ascending = False)
GOI = []
for cell in concat['Unnamed: 0'].unique():
    df__ = concat[concat['Unnamed: 0'] == cell].sort_values(by = 'posterior_mean', ascending = False)
    df__ = df__[df__['selected_fdr_0.05']]
    GOI.append(list(df__['feature'].unique()))
GOI = [item for sublist in GOI for item in sublist]
GOI = np.unique(GOI)

In [13]:
adata = sc.read_h5ad('/Users/christoffer/work/karolinska/development/oligo-mtDSB/data/mtDNA_DSB_5k_clustered_annotation_with_rbd_2.h5ad')

In [ ]:
import os
import pandas as pd

# ============ Config ============
outdir   = "../results/casecontrol_by_age_results_2"
ages     = ["21"]          # adjust to your obs values
features = GOI                   # your list of genes of interest
num_steps = 800
lr        = 0.03
seed      = 0

os.makedirs(outdir, exist_ok=True)
print(f"Running hierarchical case–control across {len(features)} genes × {len(ages)} ages...")

# Per-age combined files + resume state
age_combined = {age: os.path.join(outdir, f"casecontrol_combined_age-{age}.csv") for age in ages}
done_pairs_by_age = {}
header_written_by_age = {}

for age in ages:
    if os.path.exists(age_combined[age]):
        df_done = pd.read_csv(age_combined[age], usecols=["feature"])
        done_pairs_by_age[age] = set(df_done["feature"].unique())  # features done for this age
        header_written_by_age[age] = True
        print(f"Resuming age {age}: {len(done_pairs_by_age[age])} genes already recorded.")
    else:
        done_pairs_by_age[age] = set()
        header_written_by_age[age] = False

def already_done(age: str, gene: str, age_dir: str) -> bool:
    # check per-age combined and per-gene file
    if gene in done_pairs_by_age[age]:
        return True
    gene_safe = gene.replace("/", "_").replace(" ", "_")
    gene_path = os.path.join(age_dir, f"{gene_safe}_casecontrol.csv")
    return os.path.exists(gene_path)

for age in ages:
    print(f"\n=== Age: {age} ===", flush=True)
    ad_sub = adata[adata.obs["age"] == age].copy()

    # subfolder per age
    age_dir = os.path.join(outdir, f"age-{age}")
    os.makedirs(age_dir, exist_ok=True)

    for i, gene in enumerate(features, start=1):
        if gene not in adata.var_names:
            print(f"  [{i}/{len(features)}] {gene} → ⚠️ not in adata.var_names, skipping.", flush=True)
            continue

        if already_done(age, gene, age_dir):
            print(f"  [{i}/{len(features)}] {gene} → ⏭️ already done for age {age}, skipping.", flush=True)
            continue

        print(f"  [{i}/{len(features)}] {gene} → fitting...", flush=True)

        try:
            df_gene = make_casecontrol_df(
                ad_sub,
                feature=gene,
                condition_key="condition",
                sample_key="sample_id",
                section_key="sample_id",
                group_key="cell_class",
            )

            stats, _, _ = fit_casecontrol_single(
                df_gene,
                feature=gene,
                num_steps=num_steps,
                lr=lr,
                seed=seed,
            )

            tidy = (
                stats.reset_index()
                     .rename(columns={"index": "group"})
                     .assign(feature=gene, age=age)
            )

            gene_safe = gene.replace("/", "_").replace(" ", "_")
            gene_path = os.path.join(age_dir, f"{gene_safe}_casecontrol.csv")
            tidy.to_csv(gene_path, index=False)
            print(f"     ✅ saved → {gene_path}", flush=True)

            # append to this age's combined file
            tidy.to_csv(
                age_combined[age],
                mode="a",
                header=not header_written_by_age[age],
                index=False,
            )
            header_written_by_age[age] = True
            done_pairs_by_age[age].add(gene)

        except Exception as e:
            print(f"     ❌ {gene}: {e}", flush=True)
            continue

# Optional: build a master file from per-age combineds
master_path = os.path.join(outdir, "casecontrol_combined_all_ages.csv")
parts = []
for age in ages:
    if os.path.exists(age_combined[age]):
        parts.append(pd.read_csv(age_combined[age]))
if parts:
    pd.concat(parts, ignore_index=True).to_csv(master_path, index=False)
    print(f"\n📎 Per-age combined files:")
    for age in ages:
        print(f"   - {age_combined[age]}")
    print(f"📦 Master combined (all ages): {master_path}")
else:
    print("\n(No per-age results found to combine.)")

Running hierarchical case–control across 126 genes × 1 ages...

=== Age: 21 ===
  [1/126] Adcy1 → ⏭️ already done for age 21, skipping.
  [2/126] Adora2a → ⏭️ already done for age 21, skipping.
  [3/126] Agpat4 → ⏭️ already done for age 21, skipping.
  [4/126] Ahi1 → ⏭️ already done for age 21, skipping.
  [5/126] Aldoa → ⏭️ already done for age 21, skipping.
  [6/126] Aldoc → ⏭️ already done for age 21, skipping.
  [7/126] Anln → ⏭️ already done for age 21, skipping.
  [8/126] Anxa2 → ⏭️ already done for age 21, skipping.
  [9/126] Apod → ⏭️ already done for age 21, skipping.
  [10/126] Arc → ⏭️ already done for age 21, skipping.
  [11/126] Arrdc3 → ⏭️ already done for age 21, skipping.
  [12/126] Aspa → ⏭️ already done for age 21, skipping.
  [13/126] Atf4 → ⏭️ already done for age 21, skipping.
  [14/126] Atf5 → ⏭️ already done for age 21, skipping.
  [15/126] Atp1b2 → ⏭️ already done for age 21, skipping.
  [16/126] B2m → ⏭️ already done for age 21, skipping.
  [17/126] Bcam → ⏭️ a